# March INN Perimeter Validation

Отдельная проверка за март 2026:
- считаем ИНН, которые должны быть в отчете по Lake-логике,
- сравниваем с Excel,
- выводим лишние ИНН относительно Excel (`expected - excel`).

In [ ]:
# Импорты и настройки отображения
import re
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
# Параметры запуска (март)
month_start = '2026-03-01'
month_end = '2026-03-31'

excel_path = '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx'
excel_header = 0
inn_aliases = ['ИНН', 'inn', 'c_inn', 'INN']

In [ ]:
# Подключение к Impala
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
# Вспомогательные функции: выбор колонки и нормализация ИНН
def pick_col(columns, candidates):
    cols = list(columns)
    lower_map = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        key = str(cand).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def normalize_inn(value):
    if pd.isna(value):
        return None

    s = str(value).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None

    # Восстановление выпавшего ведущего 0
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)

    if len(s) not in (10, 12):
        return None
    return s

In [ ]:
# Шаг 1: загружаем Excel марта и строим множество ИНН
excel_raw = pd.read_excel(excel_path, header=excel_header)
excel_inn_col = pick_col(excel_raw.columns, inn_aliases)

if excel_inn_col is None:
    raise ValueError(f'В Excel не найдена колонка ИНН. Доступные: {list(excel_raw.columns)}')

excel_df = pd.DataFrame({'inn_raw': excel_raw[excel_inn_col]})
excel_df['inn_norm'] = excel_df['inn_raw'].apply(normalize_inn)
excel_inn_set = set(excel_df['inn_norm'].dropna().drop_duplicates().tolist())

print(f'Excel INN column: {excel_inn_col}')
print(f'Уникальных ИНН в Excel марта: {len(excel_inn_set)}')

In [ ]:
# Шаг 2: строим expected INN по Lake-логике (agreements + companies + agr_terms)
with imp:
    expected_lake = imp.fetch(f"""
        select
            cast(a.n_agr as string) as n_agr,
            cast(a.abs_agr_id as string) as agr_id,
            cast(a.d_valid_from as date) as agr_d_valid_from,
            cast(a.d_valid_to as date) as agr_d_valid_to,
            cast(c.c_inn as string) as c_inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
          and exists (
                select 1
                from ods_alpha.scd1_agr_terms t
                where cast(t.n_agr as string) = cast(a.n_agr as string)
                  and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                  and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
                  and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                  and upper(trim(cast(t.c_fiid_grp as string))) = 'RSHB'
                  and coalesce(t.ods_deleted_flg, '0') <> '1'
          )
    """)

if expected_lake is None:
    expected_lake = pd.DataFrame(columns=['n_agr', 'agr_id', 'agr_d_valid_from', 'agr_d_valid_to', 'c_inn'])

expected_lake = expected_lake.copy()
expected_lake['inn_norm'] = expected_lake['c_inn'].apply(normalize_inn)
expected_lake = expected_lake[expected_lake['inn_norm'].notna()].copy()
expected_inn_set = set(expected_lake['inn_norm'].dropna().drop_duplicates().tolist())

print(f'Уникальных expected ИНН по Lake-правилам: {len(expected_inn_set)}')

In [ ]:
# Шаг 3: сравниваем expected vs Excel и считаем KPI
intersect_inn = expected_inn_set & excel_inn_set
extra_vs_excel = expected_inn_set - excel_inn_set
missing_vs_expected = excel_inn_set - expected_inn_set

kpi_df = pd.DataFrame([
    {'metric': 'expected_inn_cnt', 'value': len(expected_inn_set)},
    {'metric': 'excel_inn_cnt', 'value': len(excel_inn_set)},
    {'metric': 'intersect_inn_cnt', 'value': len(intersect_inn)},
    {'metric': 'extra_vs_excel_cnt', 'value': len(extra_vs_excel)},
    {'metric': 'missing_vs_expected_cnt', 'value': len(missing_vs_expected)},
])

display(kpi_df)

In [ ]:
# Шаг 4: выводим лишние ИНН относительно Excel + детали по ним
extra_vs_excel_df = pd.DataFrame({'inn_norm': sorted(list(extra_vs_excel))})
print('Лишние ИНН относительно Excel (expected - excel):')
display(extra_vs_excel_df)

extra_details_df = (
    expected_lake[
        expected_lake['inn_norm'].isin(extra_vs_excel)
    ][['inn_norm', 'n_agr', 'agr_id', 'agr_d_valid_from', 'agr_d_valid_to']]
    .drop_duplicates()
    .sort_values(['inn_norm', 'n_agr'])
)

print('Детали по лишним ИНН (top 20 строк):')
display(extra_details_df.head(20))

In [ ]:
# Шаг 5: funnel-проверка — где именно фильтры сильнее всего срезают ИНН
with imp:
    funnel_df = imp.fetch(f"""
        with base_agreements as (
            select
                cast(a.n_agr as string) as n_agr,
                regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_raw
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
        ),
        terms_active as (
            select distinct cast(t.n_agr as string) as n_agr
            from ods_alpha.scd1_agr_terms t
            where cast(t.d_valid_from as date) <= cast('{month_end}' as date)
              and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
              and coalesce(t.ods_deleted_flg, '0') <> '1'
        ),
        terms_active_p as (
            select distinct cast(t.n_agr as string) as n_agr
            from ods_alpha.scd1_agr_terms t
            where cast(t.d_valid_from as date) <= cast('{month_end}' as date)
              and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
              and upper(trim(cast(t.cf_ter_type as string))) = 'P'
              and coalesce(t.ods_deleted_flg, '0') <> '1'
        ),
        terms_active_p_rshb as (
            select distinct cast(t.n_agr as string) as n_agr
            from ods_alpha.scd1_agr_terms t
            where cast(t.d_valid_from as date) <= cast('{month_end}' as date)
              and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
              and upper(trim(cast(t.cf_ter_type as string))) = 'P'
              and upper(trim(cast(t.c_fiid_grp as string))) = 'RSHB'
              and coalesce(t.ods_deleted_flg, '0') <> '1'
        ),
        s1 as (
            select distinct b.inn_raw
            from base_agreements b
        ),
        s2 as (
            select distinct b.inn_raw
            from base_agreements b
            join terms_active t on t.n_agr = b.n_agr
        ),
        s3 as (
            select distinct b.inn_raw
            from base_agreements b
            join terms_active_p t on t.n_agr = b.n_agr
        ),
        s4 as (
            select distinct b.inn_raw
            from base_agreements b
            join terms_active_p_rshb t on t.n_agr = b.n_agr
        )
        select '01_base_agreements_sa_active' as stage, count(*) as inn_cnt from s1
        union all
        select '02_plus_terms_active' as stage, count(*) as inn_cnt from s2
        union all
        select '03_plus_cf_ter_type_P' as stage, count(*) as inn_cnt from s3
        union all
        select '04_plus_c_fiid_grp_RSHB' as stage, count(*) as inn_cnt from s4
    """)

if funnel_df is None:
    funnel_df = pd.DataFrame(columns=['stage', 'inn_cnt'])

display(funnel_df)